# Ingredient Category Analysis

This notebook adds a broader ingredient-category analysis to reduce the sparsity problem in the individual ingredient tests. It does not replace the original hypothesis testing notebook; it complements it.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load processed dataset created by data_processing.ipynb
df = pd.read_csv('processed_culinary_climate_data.csv')
df.head()

## Create broader ingredient categories

Potato and wheat appear in very few observations, so individual ingredient tests are not reliable. To make the analysis more stable, related ingredients are grouped into broader categories.

In [ ]:
category_map = {
    'Grain_Based': ['Rice','Wheat','Bread','Bulgur','Bun','Corn','Corn Flour','Dough','Dumplings','Flour','Maize Flour','Pasta','Pastry','Rice Noodles','Rice Porridge','Rye Bread','Rye Flour','Semolina','Breadcrumbs'],
    'Meat_Based': ['Beef','Chicken','Duck','Lamb','Meat','Meatballs','Minced Meat','Pork','Sausage','Veal'],
    'Seafood_Based': ['Fish','Herring','Hilsa Fish','Mussels','Raw Fish','Salted Cod','Seafood','Cuttlefish Ink'],
    'Vegetable_Based': ['Beans','Beets','Black Beans','Cabbage','Carrots','Chickpeas','Eggplant','Fries','Kidney Beans','Lentils','Onions','Potato','Potatoes','Root Vegetables','Tomato','Tomato Sauce','Vegetables'],
    'Dairy_Based': ['Butter','Cheese','Cheese Curds','Coconut Milk','Cream','Cream Sauce','Yogurt'],
    'Spice_Sauce_Based': ['Allspice','Berbere','Herbs','Lime','Mustard Oil','Paprika','Pepper','Saffron','Salt','Soy Sauce','Spices','Sweet Soy Sauce','Vinegar','Wine','Gravy'],
    'Egg_Based': ['Egg','Eggs']
}

for category, ingredients in category_map.items():
    existing_cols = [col for col in ingredients if col in df.columns]
    df[category] = (df[existing_cols].sum(axis=1) > 0).astype(int)

category_counts = df[list(category_map.keys())].sum().sort_values(ascending=False)
category_counts

## Save category-level dataset

In [ ]:
df.to_csv('processed_culinary_climate_data_with_categories.csv', index=False)

summary_rows = []
for category, ingredients in category_map.items():
    existing_cols = [col for col in ingredients if col in df.columns]
    summary_rows.append({
        'Ingredient_Category': category,
        'Observation_Count': int(df[category].sum()),
        'Mapped_Ingredients': ', '.join(existing_cols)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Observation_Count', ascending=False)
summary_df.to_csv('ingredient_category_summary.csv', index=False)
summary_df

## Exploratory Welch tests by category

These tests compare each ingredient category with all other countries for temperature, humidity, and precipitation. Because the dataset is small, the results should be interpreted as exploratory.

In [ ]:
results = []
climate_vars = ['Temperature', 'Humidity', 'Precipitation']

for category in category_map.keys():
    for var in climate_vars:
        group_1 = df.loc[df[category] == 1, var].dropna()
        group_0 = df.loc[df[category] == 0, var].dropna()
        if len(group_1) > 1 and len(group_0) > 1:
            t_stat, p_value = stats.ttest_ind(group_1, group_0, equal_var=False)
        else:
            t_stat, p_value = np.nan, np.nan
        results.append({
            'Category': category,
            'Climate_Variable': var,
            'n_category': len(group_1),
            'n_other': len(group_0),
            'mean_category': round(group_1.mean(), 4),
            'mean_other': round(group_0.mean(), 4),
            'welch_p_value': round(p_value, 6) if not np.isnan(p_value) else np.nan
        })

tests_df = pd.DataFrame(results)
tests_df.to_csv('category_hypothesis_tests.csv', index=False)
tests_df

## Interpretation

The broader category analysis is more reliable than comparing only Potato or Wheat because the category sample sizes are larger. However, the results are still exploratory because each country is represented by only one dish and cuisine is affected by many non-climate factors.